# ML-experiment: förutsäga aktier som slår medianen

**Fråga:** Kan fem finansiella nyckeltal förutsäga om en akties avkastning *nästa år* blir högre än medianen för samma dataår?

**Upplägg**

| Steg | Data |
|---|---|
| Träning | 2014–2016 |
| Validering (modellval och inställningar) | 2017 |
| Slutligt test (körs en gång) | 2018 |

Två modeller (logistisk regression och Random Forest) jämförs mot en enkel baseline. Detta är ett experiment, inte ett investeringsråd.

## 1. Importer och inställningar

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix, roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

FEATURES = [
    "ROE",
    "PE ratio",
    "Debt to Equity",
    "Net Profit Margin",
    "Gross Margin",
]

TRAIN_YEARS, VAL_YEAR, TEST_YEAR = [2014, 2015, 2016], 2017, 2018

## 2. Läs in data

Notebooken letar efter mappen `Data` i projektmappen eller en nivå upp, så den fungerar för alla i gruppen utan att ändra sökvägar.

In [ ]:
def find_data_dir():
    for candidate in [Path("Data"), Path("../Data"), Path("../../Data")]:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Hittar inte mappen 'Data'. Lägg den i projektets rotmapp.")

DATA_DIR = find_data_dir()
print("Data-mapp:", DATA_DIR.resolve())

## 3. Målvariabel och nyckeltal

För dataåret *Y* används kolumnen `Y+1 PRICE VAR [%]` (nästa års kursförändring). 
`target = 1` betyder att avkastningen är högre än **medianen för det året**, annars `0`. Medianen räknas per år så att marknadens allmänna upp- och nedgångar inte styr målet.

Rader utan känd avkastning tas bort. I nyckeltalen görs `inf` till saknade värden, som hanteras i modellens pipeline.

In [ ]:
def load_year(year):
    df = pd.read_csv(DATA_DIR / f"{year}_Financial_Data.csv")

    return_col = f"{year + 1} PRICE VAR [%]"
    if return_col not in df.columns:
        raise KeyError(f"Kolumnen '{return_col}' saknas i filen för {year}.")
    missing = [c for c in FEATURES if c not in df.columns]
    if missing:
        raise KeyError(f"{year} saknar nyckeltal: {missing}")

    df["next_year_return"] = pd.to_numeric(df[return_col], errors="coerce")
    df = df.dropna(subset=["next_year_return"]).copy()

    median = df["next_year_return"].median()
    df["target"] = (df["next_year_return"] > median).astype(int)
    df["year"] = year

    df[FEATURES] = (
        df[FEATURES].apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
    )
    print(f"{year}: rader={len(df):>5}, median avkastning={median:7.2f} %, "
          f"andel target=1: {df['target'].mean():.3f}")
    return df[["year", *FEATURES, "next_year_return", "target"]]

data = {year: load_year(year) for year in TRAIN_YEARS + [VAL_YEAR, TEST_YEAR]}

## 4. Dela upp i träning, validering och test

Uppdelningen följer tiden: modellerna tränas på tidigare år och bedöms på senare år.

In [ ]:
train_df = pd.concat([data[y] for y in TRAIN_YEARS], ignore_index=True)
val_df = data[VAL_YEAR]
test_df = data[TEST_YEAR]

X_train, y_train = train_df[FEATURES], train_df["target"]
X_val, y_val = val_df[FEATURES], val_df["target"]
X_test, y_test = test_df[FEATURES], test_df["target"]

print(f"Träning:    {X_train.shape}")
print(f"Validering: {X_val.shape}")
print(f"Test:       {X_test.shape}")

## 5. Snabb datakontroll (endast träningsdata)

Saknade värden och extremvärden påverkar framför allt logistisk regression. Därför tittar vi på dem innan modellerna byggs.

In [ ]:
overview = X_train.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
overview["saknas (%)"] = X_train.isna().mean() * 100
overview.round(2)

In [ ]:
fig, axes = plt.subplots(1, len(FEATURES), figsize=(16, 3))
for ax, col in zip(axes, FEATURES):
    ax.boxplot(X_train[col].dropna(), showfliers=True)
    ax.set_title(col, fontsize=10)
    ax.set_xticks([])
plt.tight_layout()
plt.show()

**Tolkning:** Om percentilerna visar mycket stora skillnader mellan medianen och maxvärdet finns tydliga extremvärden. 
Logistisk regression får därför ett extra steg som begränsar värden till träningsdatans 1:a och 99:e percentil (*winsorisering*). Gränserna lärs bara från träningsdata, så ingen information läcker från validering eller test. Random Forest påverkas inte av storleken på extremvärden och får inget sådant steg.

## 6. Modeller

In [ ]:
class Winsorizer(BaseEstimator, TransformerMixin):
    """Begränsar varje kolumn till percentilgränser som lärs från träningsdata."""

    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.low_ = np.nanquantile(X, self.lower, axis=0)
        self.high_ = np.nanquantile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.low_, self.high_)


def make_baseline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DummyClassifier(strategy="most_frequent")),
    ])


def make_logreg(C=1.0):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("winsorizer", Winsorizer()),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(C=C, max_iter=2000, random_state=RANDOM_STATE)),
    ])


def make_rf(min_samples_leaf=5, max_depth=None):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=min_samples_leaf,
            max_depth=max_depth,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])


def evaluate(model, X, y):
    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        "Accuracy": accuracy_score(y, pred),
        "Balanced accuracy": balanced_accuracy_score(y, pred),
        "ROC-AUC": roc_auc_score(y, proba),
    }

## 7. Välj inställningar på valideringsdata (2017)

Ett litet rutnät av inställningar testas för varje modell. Den inställning med högst **ROC-AUC på 2017** väljs. Testdata (2018) används inte i det här steget.

In [ ]:
search_space = (
    [("Logistisk regression", {"C": C}, make_logreg(C)) for C in [0.01, 0.1, 1, 10]]
    + [
        ("Random Forest", {"min_samples_leaf": leaf, "max_depth": depth},
         make_rf(leaf, depth))
        for leaf in [5, 20, 50]
        for depth in [3, 5, None]
    ]
)

rows = []
for family, params, model in search_space:
    model.fit(X_train, y_train)
    rows.append({"Modell": family, "Inställning": str(params),
                 **evaluate(model, X_val, y_val), "_model": model})

tuning = pd.DataFrame(rows)
tuning.drop(columns="_model").sort_values(
    ["Modell", "ROC-AUC"], ascending=[True, False]
).round(4).reset_index(drop=True)

## 8. Jämför modellerna på valideringsdata (2017)

In [ ]:
best_logreg = tuning[tuning["Modell"] == "Logistisk regression"].nlargest(1, "ROC-AUC").iloc[0]
best_rf = tuning[tuning["Modell"] == "Random Forest"].nlargest(1, "ROC-AUC").iloc[0]

baseline = make_baseline().fit(X_train, y_train)

final_models = {
    "Baseline": baseline,
    "Logistisk regression": best_logreg["_model"],
    "Random Forest": best_rf["_model"],
}

print("Valda inställningar:")
print("  Logistisk regression:", best_logreg["Inställning"])
print("  Random Forest:       ", best_rf["Inställning"])

val_results = pd.DataFrame(
    {name: evaluate(model, X_val, y_val) for name, model in final_models.items()}
).T.sort_values("ROC-AUC", ascending=False)
val_results.round(4)

### Hur säkra är skillnaderna?

Skillnaderna mellan modellerna är små, så vi uppskattar osäkerheten med bootstrap (omsampling av valideringsraderna) och visar ett 95 %-intervall för ROC-AUC. Om intervallet täcker 0,5 går det inte att skilja modellen från slumpen.

In [ ]:
def bootstrap_auc(y_true, proba, n_boot=1000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        scores.append(roc_auc_score(y_true[idx], proba[idx]))
    return np.percentile(scores, [2.5, 97.5])

ci_rows = []
for name in ["Logistisk regression", "Random Forest"]:
    proba = final_models[name].predict_proba(X_val)[:, 1]
    low, high = bootstrap_auc(y_val, proba)
    ci_rows.append({"Modell": name,
                    "ROC-AUC": roc_auc_score(y_val, proba),
                    "95 % intervall": f"[{low:.3f}, {high:.3f}]"})
pd.DataFrame(ci_rows).round(4)

## 9. Vilka nyckeltal bidrar mest?

Logistisk regression: koefficienter (efter winsorisering och standardisering, så de är jämförbara). 
Random Forest: viktighet baserad på hur mycket varje nyckeltal minskar osäkerheten i träden.

In [ ]:
logreg_coef = pd.Series(
    final_models["Logistisk regression"].named_steps["model"].coef_[0], index=FEATURES
).sort_values()
rf_importance = pd.Series(
    final_models["Random Forest"].named_steps["model"].feature_importances_, index=FEATURES
).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
logreg_coef.plot(kind="barh", ax=axes[0], title="Logistisk regression: koefficient")
rf_importance.plot(kind="barh", ax=axes[1], title="Random Forest: viktighet")
plt.tight_layout()
plt.show()

## 10. Slutligt test på 2018

Modellvalet är gjort på 2017. Testet körs **en gång** som kontroll. Ändrar vi modell eller inställningar efter att ha sett 2018 blir året i praktiken ännu en valideringsmängd.

In [ ]:
test_results = pd.DataFrame(
    {name: evaluate(model, X_test, y_test) for name, model in final_models.items()}
).T.sort_values("ROC-AUC", ascending=False)
test_results.round(4)

In [ ]:
chosen_name = val_results.drop(index="Baseline")["ROC-AUC"].idxmax()
chosen_model = final_models[chosen_name]

test_pred = (chosen_model.predict_proba(X_test)[:, 1] >= 0.5).astype(int)
cm = pd.DataFrame(
    confusion_matrix(y_test, test_pred),
    index=["Verklig: under median", "Verklig: över median"],
    columns=["Förutsagd: under", "Förutsagd: över"],
)
print(f"Vald modell (högst ROC-AUC på 2017): {chosen_name}")
cm

## 11. Slutsats

In [ ]:
val_auc = val_results["ROC-AUC"]
test_auc = test_results["ROC-AUC"]
val_acc = val_results["Accuracy"]
test_acc = test_results["Accuracy"]

print(f"Vald modell enligt valideringsåret 2017: {chosen_name}")
print()
print(f"{'Modell':<22}{'Acc 2017':>10}{'AUC 2017':>10}{'Acc 2018':>10}{'AUC 2018':>10}")
for name in final_models:
    print(f"{name:<22}{val_acc[name]:>10.3f}{val_auc[name]:>10.3f}"
          f"{test_acc[name]:>10.3f}{test_auc[name]:>10.3f}")
print()

best_test = test_auc.drop(index="Baseline").idxmax()
if best_test != chosen_name:
    print(f"Observera: {best_test} var bäst på 2018, men modellvalet gjordes på 2017 "
          "och ska inte ändras efteråt.")

**Tolkning (fyll i efter körning)**

1. Hur mycket bättre än baseline är den valda modellen på 2017 och 2018?
2. Ligger ROC-AUC-intervallet i avsnitt 8 över 0,5?
3. Vilka nyckeltal väger tyngst (avsnitt 9), och är det rimligt ekonomiskt?
4. Är förbättringen tillräckligt stor för att motivera Random Forest framför den enklare logistiska regressionen?

**Begränsningar:** Bara fem nyckeltal, tre träningsår och en hög andel brus i aktieavkastning. Förbättringar på några få procentenheter ska tolkas försiktigt.